---
layout: post
title: AP CSP Pseudocode Runner
description: A tool to run AP CSP Pseudocode.
breadcrumb: True
courses: { csp: {week: 30} }
comments: false
type: ccc
permalink: /csp/pseudocode-runner/
---

# Pseudocode Runner
This notebook allows you to run AP CSP pseudocode directly in JavaScript. You can write your pseudocode in the cells below, and it will be executed as JavaScript code. This is a great way to test your understanding of pseudocode and see how it translates to actual code.




<style>
#csp-runner{font-family:system-ui,-apple-system,sans-serif;max-width:900px;margin:1.5rem auto;background:#0d1117;border:1px solid #30363d;border-radius:10px;overflow:hidden;color:#c9d1d9;font-size:14px}
#csp-runner *{box-sizing:border-box}
#csp-hdr{display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:8px;padding:12px 18px;background:#161b22;border-bottom:1px solid #30363d}
#csp-title{font-weight:600;color:#e6edf3;font-size:.95rem}
#csp-hints{display:flex;gap:10px;flex-wrap:wrap;align-items:center;font-size:.73rem;color:#8b949e}
#csp-hints kbd{background:#21262d;border:1px solid #30363d;border-radius:4px;padding:1px 5px;font-family:'Courier New',monospace;font-size:.72rem;color:#c9d1d9}
#csp-hints code{color:#ffa657;font-family:'Courier New',monospace}
#csp-edit{display:flex;height:320px;border-bottom:1px solid #30363d}
#csp-ln{background:#161b22;color:#6e7681;font-family:'Courier New',Courier,monospace;font-size:13px;line-height:1.6;padding:12px 10px 12px 14px;min-width:46px;text-align:right;border-right:1px solid #30363d;user-select:none;overflow:hidden;white-space:pre}
#csp-area{position:relative;flex:1;overflow:hidden}
#csp-hl,#csp-ta{position:absolute;top:0;left:0;right:0;bottom:0;padding:12px 14px;font-family:'Courier New',Courier,monospace;font-size:13px;line-height:1.6;white-space:pre-wrap;word-break:break-all;tab-size:3;margin:0}
#csp-hl{overflow:hidden;pointer-events:none;color:#c9d1d9;border:none;background:transparent}
#csp-ta{overflow:auto;background:transparent;color:transparent;caret-color:#58a6ff;border:none;outline:none;resize:none}
#csp-runner .hkw{color:#ff7b72;font-weight:bold}
#csp-runner .hbi{color:#79c0ff}
#csp-runner .hst{color:#a5d6ff}
#csp-runner .hnm{color:#f0883e}
#csp-runner .hop{color:#ffa657;font-weight:bold}
#csp-runner .hcm{color:#8b949e;font-style:italic}
#csp-runner .hbl{color:#56d364}
#csp-bar{display:flex;gap:8px;padding:10px 16px;background:#161b22;border-bottom:1px solid #30363d}
#csp-runner button{padding:6px 16px;border:none;border-radius:6px;cursor:pointer;font-size:.83rem;font-weight:500;transition:filter .15s}
#csp-runner button:hover{filter:brightness(1.2)}
#csp-run{background:#238636;color:#fff}
#csp-clear,#csp-ex{background:#21262d;color:#c9d1d9;border:1px solid #30363d}
#csp-outbox{padding:12px 18px 16px}
#csp-outlbl{font-size:.68rem;text-transform:uppercase;letter-spacing:.09em;color:#6e7681;margin-bottom:8px}
#csp-out{font-family:'Courier New',Courier,monospace;font-size:13px;line-height:1.6;min-height:40px;max-height:260px;overflow-y:auto;color:#c9d1d9}
#csp-runner .ol{padding:0}
#csp-runner .oerr{color:#f85149}
#csp-runner .omt{color:#6e7681;font-style:italic}
</style>

<div id="csp-runner">
<div id="csp-hdr">
  <span id="csp-title">&#9881; AP CSP Pseudocode Runner</span>
  <div id="csp-hints">
    <span>type <kbd>&lt;--</kbd> &#8594; <code>&#8592;</code></span>
    <span><kbd>!=</kbd> &#8594; <code>&#8800;</code></span>
    <span><kbd>&gt;=</kbd> &#8594; <code>&#8805;</code></span>
    <span><kbd>&lt;=</kbd> &#8594; <code>&#8804;</code></span>
    <span><kbd>Tab</kbd> = indent</span>
  </div>
</div>
<div id="csp-edit">
  <div id="csp-ln">1</div>
  <div id="csp-area">
    <pre id="csp-hl" aria-hidden="true"></pre>
    <textarea id="csp-ta" spellcheck="false" autocorrect="off" autocapitalize="off"
      placeholder="// Write AP CSP pseudocode here..."></textarea>
  </div>
</div>
<div id="csp-bar">
  <button id="csp-run">&#9654; Run</button>
  <button id="csp-clear">&#10005; Clear</button>
  <button id="csp-ex">&#8862; Example</button>
</div>
<div id="csp-outbox">
  <div id="csp-outlbl">Output</div>
  <div id="csp-out"><em class="omt">Press &#9654; Run to execute your code.</em></div>
</div>
</div>

<script>
(function(){
'use strict';

var KW = new Set([
  'IF','ELSE','REPEAT','TIMES','UNTIL','FOR','EACH','IN',
  'PROCEDURE','RETURN','AND','OR','NOT','MOD','TRUE','FALSE',
  'DISPLAY','INPUT','APPEND','INSERT','REMOVE','LENGTH'
]);

function tokenize(src) {
  var out = [], i = 0, n = src.length;
  while (i < n) {
    var c = src[i];
    if (/\s/.test(c)) { i++; continue; }
    if (c === '/' && src[i+1] === '/') { while (i < n && src[i] !== '\n') i++; continue; }
    if (c === '"' || c === '“' || c === '”') {
      var s = ''; i++;
      while (i < n && src[i] !== '"' && src[i] !== '”') s += src[i++];
      i++; out.push({t:'S',v:s}); continue;
    }
    if (c === "'") {
      var s = ''; i++; while (i < n && src[i] !== "'") s += src[i++]; i++;
      out.push({t:'S',v:s}); continue;
    }
    if (/[0-9]/.test(c)) {
      var s = ''; while (i < n && /[0-9.]/.test(src[i])) s += src[i++];
      out.push({t:'N',v:+s}); continue;
    }
    if (/[a-zA-Z_]/.test(c)) {
      var s = ''; while (i < n && /[a-zA-Z0-9_]/.test(src[i])) s += src[i++];
      var u = s.toUpperCase();
      out.push({t: KW.has(u) ? 'K' : 'I', v: KW.has(u) ? u : s}); continue;
    }
    var sl = src.slice(i);
    if (sl.slice(0,3) === '<--')  { out.push({t:'O',v:'←'}); i+=3; continue; }
    if (sl.slice(0,2) === '!=')   { out.push({t:'O',v:'≠'}); i+=2; continue; }
    if (sl.slice(0,2) === '>=')   { out.push({t:'O',v:'≥'}); i+=2; continue; }
    if (sl.slice(0,2) === '<=')   { out.push({t:'O',v:'≤'}); i+=2; continue; }
    if ('←≠≥≤'.indexOf(c) >= 0) { out.push({t:'O',v:c}); i++; continue; }
    if ('+-*/=<>(){}[],'.indexOf(c) >= 0) { out.push({t:'O',v:c}); i++; continue; }
    i++;
  }
  out.push({t:'E',v:''}); return out;
}

function Parser(toks) { this.t = toks; this.i = 0; }
Parser.prototype = {
  c:    function()  { return this.t[this.i]; },
  next: function()  { return this.t[this.i++]; },
  is:   function(t,v) { var c=this.c(); return c.t===t && (v==null||c.v===v); },
  eat:  function(t,v) {
    if (!this.is(t,v)) throw new Error('Expected '+(v||t)+", got '"+this.c().v+"'");
    return this.next();
  },
  try_: function(t,v) { if (this.is(t,v)) { this.next(); return true; } return false; },
  parse: function() { var s=[]; while(!this.is('E')) s.push(this.stmt()); return s; },
  stmt: function() {
    var c = this.c();
    if (c.t === 'K') {
      if (c.v==='IF')        return this.sIf();
      if (c.v==='REPEAT')    return this.sRep();
      if (c.v==='FOR')       return this.sFE();
      if (c.v==='PROCEDURE') return this.sProc();
      if (c.v==='RETURN')    return this.sRet();
      if (c.v==='DISPLAY')   return this.sDsp();
    }
    if (c.t === 'I') {
      var sv=this.i, name=this.next().v;
      if (this.try_('O','[')) {
        var idx=this.expr(); this.eat('O',']');
        if (this.try_('O','←')) return {k:'LA',name:name,idx:idx,val:this.expr()};
        this.i = sv;
      } else if (this.try_('O','←')) {
        return {k:'A',name:name,val:this.expr()};
      } else { this.i = sv; }
    }
    return {k:'X',e:this.expr()};
  },
  sDsp: function() { this.eat('K','DISPLAY'); this.eat('O','('); var e=this.expr(); this.eat('O',')'); return {k:'D',e:e}; },
  sIf: function() {
    this.eat('K','IF'); this.eat('O','('); var cond=this.expr(); this.eat('O',')');
    this.eat('O','{'); var then=this.block(); this.eat('O','}');
    var els=null;
    if (this.try_('K','ELSE')) { this.eat('O','{'); els=this.block(); this.eat('O','}'); }
    return {k:'IF',cond:cond,then:then,els:els};
  },
  sRep: function() {
    this.eat('K','REPEAT');
    if (this.try_('K','UNTIL')) {
      this.eat('O','('); var cond=this.expr(); this.eat('O',')');
      this.eat('O','{'); var body=this.block(); this.eat('O','}');
      return {k:'RU',cond:cond,body:body};
    }
    var cnt=this.expr(); this.eat('K','TIMES');
    this.eat('O','{'); var body=this.block(); this.eat('O','}');
    return {k:'RT',cnt:cnt,body:body};
  },
  sFE: function() {
    this.eat('K','FOR'); this.eat('K','EACH');
    var item=this.eat('I').v; this.eat('K','IN');
    var lst=this.expr(); this.eat('O','{'); var body=this.block(); this.eat('O','}');
    return {k:'FE',item:item,lst:lst,body:body};
  },
  sProc: function() {
    this.eat('K','PROCEDURE'); var name=this.eat('I').v; this.eat('O','(');
    var params=[];
    while (!this.is('O',')')) { params.push(this.eat('I').v); this.try_('O',','); }
    this.eat('O',')'); this.eat('O','{'); var body=this.block(); this.eat('O','}');
    return {k:'P',name:name,params:params,body:body};
  },
  sRet: function() {
    this.eat('K','RETURN');
    var e = (this.is('O','}')||this.is('E')) ? null : this.expr();
    return {k:'R',e:e};
  },
  block: function() { var s=[]; while(!this.is('O','}')&&!this.is('E')) s.push(this.stmt()); return s; },
  expr: function() { return this.eOr(); },
  eOr:  function() { var l=this.eAnd(); while(this.try_('K','OR'))  l={k:'B',op:'OR', l:l,r:this.eAnd()}; return l; },
  eAnd: function() { var l=this.eNot(); while(this.try_('K','AND')) l={k:'B',op:'AND',l:l,r:this.eNot()}; return l; },
  eNot: function() { if(this.try_('K','NOT')) return {k:'U',op:'NOT',o:this.eNot()}; return this.eCmp(); },
  eCmp: function() {
    var l=this.eAdd();
    var C=['≠','≥','≤','=','>','<'];
    while(this.is('O') && C.indexOf(this.c().v)>=0) { var op=this.next().v; l={k:'B',op:op,l:l,r:this.eAdd()}; }
    return l;
  },
  eAdd: function() {
    var l=this.eMul();
    while(this.is('O')&&(this.c().v==='+'||this.c().v==='-')) { var op=this.next().v; l={k:'B',op:op,l:l,r:this.eMul()}; }
    return l;
  },
  eMul: function() {
    var l=this.eUn();
    while((this.is('O')&&(this.c().v==='*'||this.c().v==='/')) || this.is('K','MOD')) {
      var op=this.next().v; l={k:'B',op:op,l:l,r:this.eUn()};
    }
    return l;
  },
  eUn: function() { if(this.is('O','-')){this.next();return{k:'U',op:'-',o:this.ePri()};} return this.ePri(); },
  ePri: function() {
    var c=this.c();
    if(c.t==='N'){this.next();return{k:'N',v:c.v};}
    if(c.t==='S'){this.next();return{k:'S',v:c.v};}
    if(c.t==='K'&&c.v==='TRUE') {this.next();return{k:'N',v:true};}
    if(c.t==='K'&&c.v==='FALSE'){this.next();return{k:'N',v:false};}
    var BI=['APPEND','INSERT','REMOVE','LENGTH','INPUT','DISPLAY'];
    if(c.t==='K'&&BI.indexOf(c.v)>=0){
      var name=this.next().v; this.eat('O','(');
      var args=[]; while(!this.is('O',')')){args.push(this.expr());this.try_('O',',');}
      this.eat('O',')'); return {k:'BI',name:name,args:args};
    }
    if(this.try_('O','(')){var e=this.expr();this.eat('O',')');return e;}
    if(this.try_('O','[')){
      var els=[]; while(!this.is('O',']')){els.push(this.expr());this.try_('O',',');}
      this.eat('O',']'); return {k:'L',els:els};
    }
    if(c.t==='I'){
      var name=this.next().v;
      if(this.try_('O','(')){
        var args=[]; while(!this.is('O',')')){args.push(this.expr());this.try_('O',',');}
        this.eat('O',')'); return {k:'C',name:name,args:args};
      }
      if(this.try_('O','[')){var idx=this.expr();this.eat('O',']');return{k:'LG',name:name,idx:idx};}
      return {k:'V',name:name};
    }
    throw new Error("Unexpected token: '"+c.v+"'");
  }
};

function gen(ast) {
  var scopes=[new Set()], depth=0, lid=0;
  function push(pre){scopes.push(new Set(pre||[]));depth++;}
  function pop(){scopes.pop();depth--;}
  function decl(n){scopes[scopes.length-1].add(n);}
  function has(n){return scopes.some(function(s){return s.has(n);});}
  function ind(){var s='';for(var i=0;i<depth;i++)s+='  ';return s;}
  // + and comparisons are delegated to __add/__cmp to eliminate JS implicit coercion
  var OPS={'AND':'&&','OR':'||','-':'-','*':'*','/':'/','MOD':'%'};
  var CMPS=new Set(['=','≠','>','<','≥','≤']);
  function stmts(ss){return ss.map(function(s){return stmt(s);}).join('\n');}
  function stmt(n){
    var p=ind();
    if(n.k==='A'){
      var v=expr(n.val);
      if(!has(n.name)){decl(n.name);return p+'let '+n.name+' = '+v+';';}
      return p+n.name+' = '+v+';';
    }
    if(n.k==='LA') return p+n.name+'[('+expr(n.idx)+')-1] = '+expr(n.val)+';';
    if(n.k==='D')  return p+'__out('+expr(n.e)+');';
    if(n.k==='IF'){
      push(); var th=stmts(n.then); pop();
      var code=p+'if ('+expr(n.cond)+') {\n'+th+'\n'+p+'}';
      if(n.els){push();var el=stmts(n.els);pop();code+=' else {\n'+el+'\n'+p+'}';}
      return code;
    }
    if(n.k==='RT'){
      var id='_i'+depth+(lid++), cnt=expr(n.cnt);
      push(); var step=ind()+'__step();\n'; var body=step+stmts(n.body); pop();
      return p+'for (let '+id+'=0; '+id+'<('+cnt+'); '+id+'++) {\n'+body+'\n'+p+'}';
    }
    if(n.k==='RU'){
      push(); var step=ind()+'__step();\n'; var body=step+stmts(n.body); pop();
      return p+'while (!('+expr(n.cond)+')) {\n'+body+'\n'+p+'}';
    }
    if(n.k==='FE'){
      var lst=expr(n.lst); push([n.item]);
      var step=ind()+'__step();\n'; var body=step+stmts(n.body); pop();
      return p+'for (let '+n.item+' of '+lst+') {\n'+body+'\n'+p+'}';
    }
    if(n.k==='P'){
      push(n.params);
      // __step() at procedure entry counts each call, catching runaway recursion
      var step=ind()+'__step();\n';
      var body=step+stmts(n.body);
      pop();
      return p+'function '+n.name+'('+n.params.join(',')+') {\n'+body+'\n'+p+'}';
    }
    if(n.k==='R') return p+'return'+(n.e?' '+expr(n.e):'')+';';
    if(n.k==='X') return p+expr(n.e)+';';
    return p+'/* ? '+n.k+' */';
  }
  function expr(n){
    if(n.k==='N') return JSON.stringify(n.v);
    if(n.k==='S') return JSON.stringify(n.v);
    if(n.k==='V') return n.name;
    if(n.k==='L') return '['+n.els.map(expr).join(',')+']';
    if(n.k==='LG') return n.name+'[('+expr(n.idx)+')-1]';
    if(n.k==='C') return n.name+'('+n.args.map(expr).join(',')+')';
    if(n.k==='B'){
      // numeric add if both sides are numbers, else string concat — no JS coercion
      if(n.op==='+') return '__add('+expr(n.l)+','+expr(n.r)+')';
      // strict equality for = and ≠; type-checked ordering for > < ≥ ≤
      if(CMPS.has(n.op)) return '__cmp('+expr(n.l)+','+JSON.stringify(n.op)+','+expr(n.r)+')';
      return '('+expr(n.l)+' '+OPS[n.op]+' '+expr(n.r)+')';
    }
    if(n.k==='U') return n.op==='NOT'?'!('+expr(n.o)+')':'-('+expr(n.o)+')';
    if(n.k==='BI'){
      var a=n.args.map(expr);
      if(n.name==='APPEND')  return '('+a[0]+'.push('+a[1]+'),'+a[0]+')';
      if(n.name==='INSERT')  return '('+a[0]+'.splice(('+a[1]+')-1,0,'+a[2]+'),'+a[0]+')';
      if(n.name==='REMOVE')  return '('+a[0]+'.splice(('+a[1]+')-1,1),'+a[0]+')';
      if(n.name==='LENGTH')  return a[0]+'.length';
      if(n.name==='INPUT')   return '__inp('+(a[0]||'"Enter a value:"')+')';
      if(n.name==='DISPLAY') return '(__out('+a[0]+'),undefined)';
    }
    return '/* ? */';
  }
  return stmts(ast);
}

function compile(src){ return gen(new Parser(tokenize(src)).parse()); }

function execute(src){
  var lines=[];
  var __out=function(v){
    if(Array.isArray(v)) lines.push('['+v.map(function(x){return JSON.stringify(x);}).join(', ')+']');
    else lines.push(String(v));
  };
  // Deterministic +: numeric addition when both operands are numbers, string concat otherwise
  var __add=function(a,b){
    if(typeof a==='number'&&typeof b==='number') return a+b;
    return String(a)+String(b);
  };
  // Type-safe comparisons: = and ≠ use strict equality; ordering ops reject non-numbers
  var __cmp=function(a,op,b){
    if(op==='=') return a===b;
    if(op==='≠') return a!==b;
    if(typeof a!=='number'||typeof b!=='number')
      throw new Error('AP CSP Runtime Error: Cannot use "'+op+'" to compare non-numeric values');
    if(op==='>') return a>b;
    if(op==='<') return a<b;
    if(op==='≥') return a>=b;
    return a<=b;
  };
  // Input type inference: int → float → bool (case-insensitive, trimmed) → string
  // Float regex requires at least one digit before or after the decimal (rejects bare ".")
  var __inp=function(msg){
    try{
      var v=(window.prompt(msg)||'').trim();
      if(/^-?\d+$/.test(v)) return parseInt(v,10);
      if(/^-?(\d+\.\d*|\d*\.\d+)$/.test(v)) return parseFloat(v);
      if(v.toUpperCase()==='TRUE') return true;
      if(v.toUpperCase()==='FALSE') return false;
      return v;
    }catch(e){return'';}
  };
  var steps=0;
  var __step=function(){
    if(++steps>100000) throw new Error('AP CSP Runtime Error: Possible infinite loop (exceeded 100k steps)');
  };
  var code=compile(src);
  try{
    new Function('__out','__inp','__step','__add','__cmp',code)(__out,__inp,__step,__add,__cmp);
  }catch(e){
    var msg=e.message||String(e);
    // Pass through already-structured errors; wrap raw JS errors with AP CSP prefix
    if(msg.indexOf('AP CSP Runtime Error:')===0) throw e;
    throw new Error('AP CSP Runtime Error: '+msg);
  }
  return lines;
}

var HKW  = new Set(['IF','ELSE','REPEAT','TIMES','UNTIL','FOR','EACH','IN','PROCEDURE','RETURN','AND','OR','NOT','MOD']);
var HBI  = new Set(['DISPLAY','INPUT','APPEND','INSERT','REMOVE','LENGTH']);
var HBOOL= new Set(['TRUE','FALSE']);

function hesc(s){return s.replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;');}

function highlight(code){
  var out='',i=0,n=code.length;
  while(i<n){
    var c=code[i];
    if(c==='\n'){out+='\n';i++;continue;}
    if(c==='/'&&code[i+1]==='/'){
      var s='';while(i<n&&code[i]!=='\n')s+=code[i++];
      out+='<span class="hcm">'+hesc(s)+'</span>';continue;
    }
    if(c==='"'||c==='“'||c==='”'||c==="'"){
      var cl=c==='"'?'"':c; var s=c; i++;
      while(i<n&&code[i]!==cl)s+=code[i++];
      s+=(code[i]||'');i++;
      out+='<span class="hst">'+hesc(s)+'</span>';continue;
    }
    if(/[0-9]/.test(c)){
      var s='';while(i<n&&/[0-9.]/.test(code[i]))s+=code[i++];
      out+='<span class="hnm">'+hesc(s)+'</span>';continue;
    }
    if(/[a-zA-Z_]/.test(c)){
      var s='';while(i<n&&/[a-zA-Z0-9_]/.test(code[i]))s+=code[i++];
      var u=s.toUpperCase();
      if(HKW.has(u))        out+='<span class="hkw">'+hesc(s)+'</span>';
      else if(HBI.has(u))   out+='<span class="hbi">'+hesc(s)+'</span>';
      else if(HBOOL.has(u)) out+='<span class="hbl">'+hesc(s)+'</span>';
      else                   out+=hesc(s);
      continue;
    }
    if('←≠≥≤'.indexOf(c)>=0){out+='<span class="hop">'+hesc(c)+'</span>';i++;continue;}
    out+=hesc(c);i++;
  }
  return out;
}

var ta  = document.getElementById('csp-ta');
var hl  = document.getElementById('csp-hl');
var ln  = document.getElementById('csp-ln');
var out = document.getElementById('csp-out');

function update(){
  hl.innerHTML = highlight(ta.value)+'\n';
  var n=(ta.value.match(/\n/g)||[]).length+1;
  var nums=''; for(var i=1;i<=n;i++) nums+=(i<n?i+'\n':i);
  ln.textContent=nums;
}

ta.addEventListener('input',function(){
  var pos=this.selectionStart, val=this.value;
  var before=val.slice(0,pos), after=val.slice(pos);
  var subs=[['<--','←',3],['!=','≠',2],['>=','≥',2],['<=','≤',2]];
  for(var i=0;i<subs.length;i++){
    var find=subs[i][0],rep=subs[i][1],len=subs[i][2];
    if(before.slice(-len)===find){
      var nb=before.slice(0,-len)+rep;
      this.value=nb+after; this.setSelectionRange(nb.length,nb.length); break;
    }
  }
  update();
});

ta.addEventListener('scroll',function(){
  hl.scrollTop=ta.scrollTop; hl.scrollLeft=ta.scrollLeft; ln.scrollTop=ta.scrollTop;
});

ta.addEventListener('keydown',function(e){
  if(e.key==='Tab'){
    e.preventDefault();
    var s=ta.selectionStart,end=ta.selectionEnd;
    ta.value=ta.value.slice(0,s)+'   '+ta.value.slice(end);
    ta.setSelectionRange(s+3,s+3); update();
  }
});

document.getElementById('csp-run').addEventListener('click',function(){
  out.innerHTML='';
  try{
    var lines=execute(ta.value.trim());
    if(!lines.length) out.innerHTML='<em class="omt">No output.</em>';
    else out.innerHTML=lines.map(function(l){return'<div class="ol">'+hesc(l)+'</div>';}).join('');
  }catch(e){
    out.innerHTML='<div class="oerr">&#9888; '+hesc(e.message)+'</div>';
  }
});

document.getElementById('csp-clear').addEventListener('click',function(){
  out.innerHTML='<em class="omt">Press &#9654; Run to execute your code.</em>';
});

/* Full feature demo - covers every AP CSP construct */
var EXAMPLE = [
  '// AP CSP Full Feature Demo',
  '// Covers: variables, arithmetic, booleans, IF/ELSE,',
  '// REPEAT TIMES, REPEAT UNTIL, FOR EACH, lists,',
  '// procedures, recursion',
  '',
  '// 1. Variables & Arithmetic',
  'a ← 10',
  'b ← 3',
  'DISPLAY("--- Arithmetic ---")',
  'DISPLAY(a + b)',
  'DISPLAY(a - b)',
  'DISPLAY(a * b)',
  'DISPLAY(a / b)',
  'DISPLAY(a MOD b)',
  '',
  '// 2. Boolean & Comparison',
  'DISPLAY("--- Booleans ---")',
  'DISPLAY(a > b)',
  'DISPLAY(a = b)',
  'DISPLAY(NOT (a = b))',
  'DISPLAY(a > 5 AND b < 5)',
  'DISPLAY(a < 5 OR b < 5)',
  '',
  '// 3. IF / ELSE',
  'DISPLAY("--- IF/ELSE ---")',
  'IF (a > b)',
  '{',
  '   DISPLAY("a is greater than b")',
  '}',
  'ELSE',
  '{',
  '   DISPLAY("b is greater or equal")',
  '}',
  '',
  '// 4. REPEAT n TIMES',
  'DISPLAY("--- REPEAT TIMES ---")',
  'sum ← 0',
  'REPEAT 5 TIMES',
  '{',
  '   sum ← sum + 1',
  '   DISPLAY(sum)',
  '}',
  '',
  '// 5. REPEAT UNTIL',
  'DISPLAY("--- REPEAT UNTIL ---")',
  'x ← 1',
  'REPEAT UNTIL (x > 50)',
  '{',
  '   DISPLAY(x)',
  '   x ← x * 2',
  '}',
  '',
  '// 6. Lists',
  'DISPLAY("--- Lists ---")',
  'nums ← [10, 20, 30, 40, 50]',
  'DISPLAY(nums[1])',
  'DISPLAY(nums[3])',
  'DISPLAY(LENGTH(nums))',
  'APPEND(nums, 60)',
  'DISPLAY(LENGTH(nums))',
  'INSERT(nums, 1, 5)',
  'DISPLAY(nums[1])',
  'REMOVE(nums, 2)',
  'DISPLAY(nums[1])',
  'nums[3] ← 999',
  'DISPLAY(nums[3])',
  '',
  '// 7. FOR EACH',
  'DISPLAY("--- FOR EACH ---")',
  'words ← ["alpha", "beta", "gamma"]',
  'FOR EACH w IN words',
  '{',
  '   DISPLAY(w)',
  '}',
  '',
  '// 8. PROCEDURE with RETURN',
  'DISPLAY("--- PROCEDURE ---")',
  'PROCEDURE max(x, y)',
  '{',
  '   IF (x > y)',
  '   {',
  '      RETURN x',
  '   }',
  '   ELSE',
  '   {',
  '      RETURN y',
  '   }',
  '}',
  'DISPLAY(max(7, 12))',
  'DISPLAY(max(99, 3))',
  '',
  '// 9. Recursive PROCEDURE',
  'DISPLAY("--- RECURSION ---")',
  'PROCEDURE factorial(n)',
  '{',
  '   IF (n ≤ 1)',
  '   {',
  '      RETURN 1',
  '   }',
  '   ELSE',
  '   {',
  '      RETURN n * factorial(n - 1)',
  '   }',
  '}',
  'DISPLAY(factorial(5))',
  'DISPLAY(factorial(10))',
  '',
  '// 10. Build a list with a loop',
  'DISPLAY("--- BUILD LIST ---")',
  'squares ← []',
  'i ← 1',
  'REPEAT 6 TIMES',
  '{',
  '   APPEND(squares, i * i)',
  '   i ← i + 1',
  '}',
  'FOR EACH s IN squares',
  '{',
  '   DISPLAY(s)',
  '}'
].join('\n');

document.getElementById('csp-ex').addEventListener('click',function(){
  ta.value=EXAMPLE; update(); ta.focus();
});

update();

})();
</script>